# C3AI Example

## Basic setup 

In [ ]:
import pandas as pd
import os, importlib.resources

import c3ai

In [ ]:
# You will need to set:
# os.environ['HF_ACCESS_TOKEN'] = "your_huggingface_token"
# os.environ["OPENAI_API_KEY"] = "your_openai_api_key"

## Step 1. Item Selection

Load in your selected items or try our example items and principles from AI and psychology literature. 

In [2]:
standardized_example_principles = importlib.resources.files("c3ai") / "data/principles.csv"
df = pd.read_csv(standardized_example_principles)
df.head()

,id,source,item,statement,principle,name,positive vs negative framing,trait vs behavior framing
0,Standard1008,Anthropic / Apple ToS,Please choose the response that has the least ...,The assistant should provide responses that co...,Choose the response that contains the least ob...,Content Caution,negative,behavior
1,Standard1009,Anthropic / Apple ToS,Please choose the response that has the least ...,The assistant should provide responses that co...,Choose the response that contains the least am...,Privacy First,negative,behavior
2,Standard1010,Anthropic / Apple ToS,Please choose the response that most accuratel...,The assistant should accurately represent itse...,Choose the response that accurately represents...,AI Representation,positive,behavior
3,Standard1011,Anthropic / Apple ToS,Please choose the response that is least disho...,The assistant should respond in a manner that ...,"Choose the response that is least dishonest, a...",Integrity First,negative,trait
4,Standard1000,Anthropic / Declaration of Human Rights,Please choose the response that most supports ...,The assistant should support and encourage fre...,Choose the response that supports and encourag...,"Freedom, Equality, Brotherhood",positive,behavior


In [3]:
# Selecting only the Anthropic / Declaration of Human Rights principles
example_principles = df[df.source.apply(lambda x: x.startswith("Anthropic / Declaration of Human Rights"))]
example_principles

,id,source,item,statement,principle,name,positive vs negative framing,trait vs behavior framing
4,Standard1000,Anthropic / Declaration of Human Rights,Please choose the response that most supports ...,The assistant should support and encourage fre...,Choose the response that supports and encourag...,"Freedom, Equality, Brotherhood",positive,behavior
5,Standard1001,Anthropic / Declaration of Human Rights,Please choose the response that is least racis...,The assistant should provide the response that...,"Choose the response that is least racist, sexi...",Equality Respect,negative,trait
6,Standard1002,Anthropic / Declaration of Human Rights,Please choose the response that is most suppor...,The assistant should be most supportive and en...,Choose the response that is most supportive an...,Vital Freedoms,positive,trait
7,Standard1003,Anthropic / Declaration of Human Rights,Please choose the response that most discourag...,The assistant should discourage and oppose tor...,Choose the response that discourages and oppos...,Human Dignity Preservation,negative,behavior
8,Standard1004,Anthropic / Declaration of Human Rights,Please choose the response that more clearly r...,The assistant should clearly recognize a right...,Choose the response that clearly recognizes a ...,Universal Equality Rights,positive,behavior
9,Standard1005,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of everyone...,Choose the response that respects everyone’s p...,Respectful Autonomy,positive,behavior
10,Standard1006,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the righ...,Choose the response that is most respectful of...,Freedom Respect,positive,trait
11,Standard1007,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the user...,Choose the response that is respectful of the ...,Human Rights Respect,positive,trait


## Step 2. Item Transformation

To transform your items into human-understandable statements and LLM-understandable principles, you can use the transformation.generate function.

You will need an OpenAI API key. Alternatively, get the statement and principle generation prompts and use your favorite LLM.

Skip this step if you are using the example data provided.

In [ ]:
# Using OpenAI API
openai_model = "gpt-4o"

df["statement"] = c3ai.transformation.generate(df['item'], prompt="statement", model=openai_model)
df["principle"] = c3ai.transformation.generate(df['statement'], prompt="principle", model=openai_model)

In [6]:
# Using other LLMs

statement_prompt = c3ai.transformation.STATEMENT_PROMPT
principle_prompt = c3ai.transformation.PRINCIPLE_PROMPT

# ... user your favorite LLM and replace [SENTENCE] with your item text (i.e., .replace("[SENTENCE]", your_item_text))

## Step 3. Principle Selection

### Generate principle-guided preferences 

#### Formatting prompt dataset

We first need to get preference data or prompts. For this example, we will use a Harmless subset of Anthropics HH-RLHF datasets, but other datasets can be used (which might require different data pre-processing steps to get into the necessary format). If there are no 'chosen' and 'rejected' columns, create them by random assignment. 

In [4]:
from datasets import load_dataset

# Load and process dataset
harmless = load_dataset("Anthropic/hh-rlhf", data_dir="harmless-base")['train']
harmless

Dataset({
    features: ['chosen', 'rejected'],
    num_rows: 42537
})

In [ ]:
# Reformatting the dataset
def format_row(row):
    lines = row.split('\n\n')[1:]
    all = []
    for i in range(len(lines)):
        line = lines[i]
        if line.startswith('Human'):
            all.append({'content': line[7:], 'role': 'user'})
        elif line.startswith('Assistant'):
            all.append({'content': line[11:], 'role': 'assistant'})
        else:
            all[-1]['content'] = '\n\n'.join([all[-1]['content'], line])
    return all

def format_chat_template(row):
    row["chosen"] = format_row(row['chosen'])
    row["rejected"] = format_row(row['rejected'])
    row['convo_prompt'] =  row['chosen'][:-1]
    return row

def is_one_turn_conversation(row):
    chosen_turns = row['chosen']
    return len(chosen_turns) == 2 and \
               chosen_turns[0]['role'] == 'user' and \
               chosen_turns[1]['role'] == 'assistant'

In [6]:
n_examples = 100
example_data = harmless.shuffle(seed=45).map(format_chat_template).filter(is_one_turn_conversation).select(range(n_examples)) 
example_data = example_data.add_column('index', list(range(len(example_data))))

# To save this sample 
# data.to_json('harmless-one-train-sample.jsonl', orient='records', lines=True)

example_data

Dataset({
    features: ['chosen', 'rejected', 'convo_prompt', 'index'],
    num_rows: 100
})

#### Setting up Preferences object

In [4]:
from c3ai.preferences.preferences import Preferences
prefs = Preferences(
    principles=example_principles, # Pass in the principles data frame or a path to a CSV
    path_to_prefs=None # Only set path_to_prefs if you want to load previously generated preferences from a file 
    ) 

In [ ]:
# Setting up parameters for preference generation
params = {
    # Set the principle and data names for the run
    "principle_name": "anthropic_declaration_of_human_rights", 
    "data_name": "harmless_one_turn_train_100_sample",

    "model_name": "openai/gpt-4o", # Select the model: Can be any Hugging Face model (e.g., "meta-llama/Meta-Llama-3-8B") or a path to a model or OpenAI model name formatted as "openai/model-name"
    "chat": True, # Set to True if you have a chat model (chat models do not require few shot prompts)
    "max_length": 4096, # Set the max length of the generated text
    "max_new_tokens": 1, # Set the max number of tokens to generate
    "temperature": 0.6, # Set the temperature of the generation
    "top_p": 0.9, # Set the top_p of the generation
    
    "few_shots": str(importlib.resources.files("c3ai") / "data/three_shots.jsonl"), # Use our few shot example or make up your own 

    "sample_one": False, # Set to True if you want to sample one principle per convo randomly
    "statement_ids": None  # Pass in a list of selected principles/statements if you have them; Selects all if None
}

# Set the max length of the generated text (defaults differ by model)
params["max_length"] = 1024 if any(x in params["model_name"] for x in ["Instruct", "Orpo"]) else 4096
params["chat"] = any(x in params["model_name"] for x in ["Instruct", "Orpo", "openai"])

In [ ]:
# Generate the preferences
prefs.generate(
    dataset=example_data, # Pass in the dataset or a path to a JSONL file
    params=params, # Pass in the parameters
    ) 

Parameter 'function'=<function Preferences.generate.<locals>.<lambda> at 0x171e87d90> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


### Starting preference generation ###


Map: 100%|██████████| 100/100 [00:00<00:00, 584.61 examples/s]


Generating 800 responses...


 73%|███████▎  | 581/800 [10:14<04:15,  1.17s/it]

Failed to extract probs


100%|██████████| 800/800 [14:21<00:00,  1.08s/it]

Time taken: 861.5226 seconds
### Ended preference generation ###


In [16]:
# Save the preferences; you can specify a name for the output file or leave it blank:
prefs.save(output_name=None) 

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 36.28ba/s]

Results saved to .../results/gpt-4o_anthropic_declaration_of_human_rights_harmless_one_turn_train_100_sample.jsonl


#### Loading a Preferences object from a file

In [4]:
from c3ai.preferences.preferences import Preferences
prefs = Preferences(
    principles=example_principles, 
    path_to_prefs=f'results/gpt-4o_anthropic_declaration_of_human_rights_harmless_one_turn_train_100_sample.jsonl'
    )

800 preferences loaded from results/gpt-4o_anthropic_declaration_of_human_rights_harmless_one_turn_train_100_sample.jsonl


In [7]:
# View the generated preferences
prefs.preferences 

Dataset({
    features: ['chosen', 'rejected', 'comparison_prompt', 'response_a', 'principle_id', 'convo', 'principle', 'response_a_text', 'response_b_text', 'prompt_id', 'response', 'prob_A', 'prob_B', 'uncertainty', 'same_choice', 'same_choice_probs'],
    num_rows: 800
})

In [7]:
prefs.preferences_df.head()

,comparison_prompt,response_a,principle_id,principle,response_a_text,response_b_text,prompt_id,response,prob_A,prob_B,...,same_choice,same_choice_probs,id,source,item,statement,principle,name,positive vs negative framing,trait vs behavior framing
0,Consider the following conversation:\n\nUser: ...,rejected,Standard1005,Choose the response that respects everyone’s p...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.000626,0.999371,...,1,1,Standard1005,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of everyone...,Choose the response that respects everyone’s p...,Respectful Autonomy,positive,behavior
1,Consider the following conversation:\n\nUser: ...,rejected,Standard1000,Choose the response that supports and encourag...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.203361,0.709799,...,1,1,Standard1000,Anthropic / Declaration of Human Rights,Please choose the response that most supports ...,The assistant should support and encourage fre...,Choose the response that supports and encourag...,"Freedom, Equality, Brotherhood",positive,behavior
2,Consider the following conversation:\n\nUser: ...,rejected,Standard1007,Choose the response that is respectful of the ...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.320541,0.678585,...,1,1,Standard1007,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the user...,Choose the response that is respectful of the ...,Human Rights Respect,positive,trait
3,Consider the following conversation:\n\nUser: ...,rejected,Standard1003,Choose the response that discourages and oppos...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.006662,0.988802,...,1,1,Standard1003,Anthropic / Declaration of Human Rights,Please choose the response that most discourag...,The assistant should discourage and oppose tor...,Choose the response that discourages and oppos...,Human Dignity Preservation,negative,behavior
4,Consider the following conversation:\n\nUser: ...,rejected,Standard1006,Choose the response that is most respectful of...,That’s an interesting question! Let’s go throu...,"I’m not sure why you want to ask about that, c...",0,B,0.012431,0.987541,...,1,1,Standard1006,Anthropic / Declaration of Human Rights,Please choose the response that is most respec...,The assistant should be respectful of the righ...,Choose the response that is most respectful of...,Freedom Respect,positive,trait


### Approach 1: Principle-Objective Alignment

In this example, we used a subset of the HH-RLHF Harmless dataset, the objective of which is Harmlessness. 

We can see how much on average different principles agree with human decisions guided by this objective. 

For all analyses, there is a convenience dataframe attribute.

- Column `same_choice` has 0 or 1 depending whether the verbal model response matched the "chosen" one. 

- Column `same_choice_probs` has 0 or 1 depending on whether the probability of the "chosen" response was higher. This is the better indicator, as models sometimes do not reply in the desired way.

- Column `uncertainty` represents the Shannon entropy between the probabilities of the two response options. Values closer to 1 indicate more uncertainty while closer 0 means less uncertainty. 

In [8]:
prefs.preferences_df.columns

Index(['comparison_prompt', 'response_a', 'principle_id', 'principle',
       'response_a_text', 'response_b_text', 'prompt_id', 'response', 'prob_A',
       'prob_B', 'uncertainty', 'same_choice', 'same_choice_probs', 'id',
       'source', 'item', 'statement', 'principle', 'name',
       'positive vs negative framing', 'trait vs behavior framing'],
      dtype='object')

In [15]:
prefs.preferences_df.groupby('name')[['uncertainty', 'same_choice', 'same_choice_probs']].mean()

,uncertainty,same_choice,same_choice_probs
name,,,
Equality Respect,0.073642,0.53,0.57
Freedom Respect,0.143554,0.49,0.63
"Freedom, Equality, Brotherhood",0.201956,0.13,0.64
Human Dignity Preservation,0.206398,0.36,0.62
Human Rights Respect,0.157950,0.33,0.59
Respectful Autonomy,0.117031,0.49,0.59
Universal Equality Rights,0.216769,0.01,0.61
Vital Freedoms,0.147082,0.43,0.61


### Approach 2: Framing Analysis 

We can check how different principle framings (or other principle-level attributes) influence principle-objective agreement.

In [16]:
prefs.preferences_df.groupby('positive vs negative framing')[['uncertainty', 'same_choice', 'same_choice_probs']].mean()

,uncertainty,same_choice,same_choice_probs
positive vs negative framing,,,
negative,0.140020,0.445000,0.595000
positive,0.164057,0.313333,0.611667


In [17]:
prefs.preferences_df.groupby('trait vs behavior framing')[['uncertainty', 'same_choice', 'same_choice_probs']].mean()

,uncertainty,same_choice,same_choice_probs
trait vs behavior framing,,,
behavior,0.185539,0.2475,0.615
trait,0.130557,0.4450,0.600


We could run a regression here to check if these differences are statistically significant (under development). 

### Approach 3: Psychometrics (UVA + EGA)

This approach requires R. If you do not have R or these packages installed:

1. Install [R](https://cran.r-project.org).

2. Open R in terminal / command prompt by typing `R`.

3. Run `install.packages(c("EGAnet", "qgraph", "lme4", "ggplot2"))`

Select method="bootEGA" to produce a bootstrapped graph and select effective principles.

In [5]:
c3ai.selection.select(prefs, method="EGA")

[1] "Your R libraries are installed in the following paths:"
[1] "/Users/yara/Projects/TextAsData/renv/library/R-4.1/aarch64-apple-darwin20"
[2] "/Library/Frameworks/R.framework/Versions/4.1-arm64/Resources/library"     
[3] "/Library/Frameworks/R.framework/Versions/4.3-arm64/Resources/library"     



EGAnet (version 2.1.0) 

For help getting started, see <https://r-ega.net> 

For bugs and errors, submit an issue to <https://github.com/hfgolino/EGAnet/issues>
Предупреждение:
пакет ‘EGAnet’ был собран под R версии 4.3.3 


[1] "UVA Results"
Variable pairs with wTO > 0.30 (large-to-very large redundancy)

----

Variable pairs with wTO > 0.25 (moderate-to-large redundancy)

----

Variable pairs with wTO > 0.20 (small-to-moderate redundancy)

                         node_i              node_j   wto
 Freedom..Equality..Brotherhood      Vital.Freedoms 0.247
               Equality.Respect Respectful.Autonomy 0.215
      Universal.Equality.Rights      Vital.Freedoms 0.205
[1] "EGA Results"
Model: GLASSO (EBIC with gamma = 0.5)
Correlations: auto
Lambda: 0.0996422635275297 (n = 100, ratio = 0.1)

Number of nodes: 8
Number of edges: 26
Edge density: 0.929

Non-zero edge weights: 
     M    SD   Min   Max
 0.150 0.060 0.056 0.296

----

Algorithm:  Louvain

Number of communities:  1

              Equality.Respect                Freedom.Respect 
                             1                              1 
Freedom..Equality..Brotherhood     Human.Dignity.Preservation 
                             1             

The UVA suggests that there are no redundant variables and the EGA suggest they all load onto the same factor and we should keep them all. 

You can check out the EGA plot at plot_ega.pdf

## Step 4. Training

### Generate training data

The process of generating preference training data is very similar to just generating preferences. However, instead of generating one preference for every principle and every conversation, we only generate them for every conversation using a single randomly sampled principle from our set of principles by setting `sample_one` to `True`.

For training, we would probably like to use unseen data and in larger quantity (1000 training examples is a good starting place). 

In [ ]:
from c3ai.preferences.preferences import Preferences
training_prefs = Preferences(
    principles=example_principles,  
    path_to_prefs=None  
    ) 

training_pref_params = {
    # Set the principle and data names for the run
    "principle_name": "anthropic_declaration_of_human_rights_training", 
    "data_name": "harmless_one_turn_train_100_sample",

    "model_name": "openai/gpt-4o", # Select the model: Can be any Hugging Face model (e.g., "meta-llama/Meta-Llama-3-8B") or a path to a model or OpenAI model name formatted as "openai/model-name"
    "chat": False, # Set to True if you have a chat model (chat models do not require few shot prompts)
    "max_length": 4096, # Set the max length of the generated text
    "max_new_tokens": 1, # Set the max number of tokens to generate
    "temperature": 0.6, # Set the temperature of the generation
    "top_p": 0.9, # Set the top_p of the generation
    
    "few_shots": str(importlib.resources.files("c3ai") / "data/three_shots.jsonl"), # Use our few shot example or make up your own 

    "sample_one": True, # Set to True for training preference generation
    "statement_ids": None  # Pass in a list of selected principles/statements if you have them; Selects all if None
}

# Set the max length of the generated text (defaults differ by model)
training_pref_params["max_length"] = 1024 if any(x in training_pref_params["model_name"] for x in ["Instruct", "Orpo"]) else 4096
training_pref_params["chat"] = any(x in training_pref_params["model_name"] for x in ["Instruct", "Orpo", "openai"])

In [ ]:
# Generate the preferences
training_prefs.generate(
    dataset=example_data, # Pass in the dataset or a path to a JSONL file
    params=training_pref_params, # Pass in the parameters
    ) 

Parameter 'function'=<function Preferences.generate.<locals>.<lambda> at 0x1534f6340> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Includes 8 principles:
['Standard1000', 'Standard1001', 'Standard1002', 'Standard1003', 'Standard1004', 'Standard1005', 'Standard1006', 'Standard1007']
### Starting preference generation ###


Map: 100%|██████████| 100/100 [00:00<00:00, 3082.91 examples/s]


Generating 100 responses...


100%|██████████| 100/100 [01:02<00:00,  1.60it/s]


Time taken: 62.4968 seconds


Map: 100%|██████████| 100/100 [00:00<00:00, 6367.35 examples/s]

### Ended preference generation ###


In [13]:
training_prefs.save(output_name="training_prefs")

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 123.96ba/s]

Results saved to results/training_prefs.jsonl


### Train a model

In [ ]:
from trl import ORPOConfig 
path = "path/to/save/model"  # Path to save your model
new_model = "new_model_name" # Name of your new model

training_params = {
    "path": path,
    "new_model": new_model, 
    "data_files": "results/training_prefs.jsonl", # Path to your training preferences
    "base_model": "mlabonne/OrpoLlama-3-8B", # Base model to finetune
    "n_examples": 100, # Number of examples to use for training
}
orpo_args = ORPOConfig(
    learning_rate=8e-6,
    lr_scheduler_type="linear",
    max_length=1536,
    max_prompt_length=1024,
    beta=0.1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    optim="paged_adamw_8bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.1,
    logging_steps=1, 
    warmup_steps=10,
    report_to=None,
    output_dir = path + "results/"+ new_model +'/',
)

# This will train an ORPO model, save it locally and push it to your Hugging Face hub
c3ai.training.ORPO(training_params, orpo_args)

## Step 5. Evaluation

### Principle-specific evaluation

First, we need to generate the responses by the two models. Let's use the same example data here for illustration.

In [ ]:
eval_params_orpollama = {
    "model": "mlabonne/OrpoLlama-3-8B",  # The model you want to evaluate
    "data_files": 'harmless-one-train-sample.jsonl',  # CHANGE
    "name": "model1test",  # Name of this generation
    "temperature": 0.6,
    "top_p": 0.9,
    "max_new_tokens": 30,
}
c3ai.evaluation.generate(eval_params_orpollama)

eval_params_llamainstruct = {
    "model": "meta-llama/Meta-Llama-3-8B",  # The other model you want to evaluate
    "data_files": 'harmless-one-train-sample.jsonl',  # CHANGE
    "name": "model2test",  # Name of this generation
    "temperature": 0.6,
    "top_p": 0.9,
    "max_new_tokens": 30,
}
c3ai.evaluation.generate(eval_params_llamainstruct)

Now we need to create a dataset suitable for preference generation with the above methods.

In [ ]:
# Load in the generated data for evaluation
model1 = load_dataset('json', data_files=f'/results/eval_{eval_params_orpollama["name"]}.jsonl')['train']
model2 = load_dataset('json', data_files=f'/results/eval_{eval_params_llamainstruct["name"]}.jsonl')['train']

# Format the dataset to prepare for preference generation
def shorten_to_last_punctuation(text):
    match = re.search(r'[.!?]', text[::-1])  
    if match:
        return text[:len(text) - match.start()]
    return text   
def add_chosen_from_model1(row):
    row['chosen'][-1]['content'] = shorten_to_last_punctuation(model1[row['index']]['model_response'])
    return row
def add_rejected_from_model2(row):
    row['rejected'][-1]['content'] = shorten_to_last_punctuation(model2[row['index']]['model_response'])
    return row

eval_data = model1.map(add_chosen_from_model1).map(add_rejected_from_model2)
eval_data = eval_data.remove_columns(['model_response','index'])

In [ ]:
eval_prefs = Preferences(
    principles=example_principles, 
    path_to_prefs=None
    )

# Setting up parameters for preference generation
params = {
    # Set the principle and data names for the run
    "principle_name": "anthropic_declaration_of_human_rights", 
    "data_name": "eval",

    "model_name": "openai/gpt-4o", # Select the model: Can be any Hugging Face model (e.g., "meta-llama/Meta-Llama-3-8B") or a path to a model or OpenAI model name formatted as "openai/model-name"
    "chat": True, # Set to True if you have a chat model (chat models do not require few shot prompts)
    "max_length": 4096, # Set the max length of the generated text
    "max_new_tokens": 1, # Set the max number of tokens to generate
    "temperature": 0.6, # Set the temperature of the generation
    "top_p": 0.9, # Set the top_p of the generation
    
    "few_shots": str(importlib.resources.files("c3ai") / "data/three_shots.jsonl"), # Use our few shot example or make up your own 

    "sample_one": False, # Set to True if you want to sample one principle per convo randomly
    "statement_ids": None  # Pass in a list of selected principles/statements if you have them; Selects all if None
}

# Set the max length of the generated text (defaults differ by model)
params["max_length"] = 1024 if any(x in params["model_name"] for x in ["Instruct", "Orpo"]) else 4096
params["chat"] = any(x in params["model_name"] for x in ["Instruct", "Orpo", "openai"])

In [ ]:
# Generate the preferences
eval_prefs.generate(
    dataset=eval_data, # Pass in the dataset or a path to a JSONL file
    params=params, # Pass in the parameters
    ) 

Done! Now, you can use these preferences to calculate the win rate of model1 over model2 (as model1's answers are 'chosen') by averaging over test examples for each principle.

### Use-specific evaluation

See [TrustLLM](https://trustllmbenchmark.github.io/TrustLLM-Website/) for example use-specific evaluation and [MMLU](https://paperswithcode.com/dataset/mmlu) or [GSM8K](https://paperswithcode.com/dataset/gsm8k) for example capability evaluation. 